  # ai question case

1. Read Pinecone index
2. For a query, find relevant documents
3. Get the answer

In [5]:
import openai
import os
from langchain.vectorstores import Pinecone
from langchain.chains.question_answering import load_qa_chain
from dotenv import load_dotenv, find_dotenv
import pinecone

_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key  = os.getenv('OPENAI_API_KEY')

In [2]:
namespace = "31"
model = "gpt-4"
index_name = os.getenv('PINECONE_INDEX_NAME')

# initialize pinecone
pinecone.init(

    api_key=os.getenv('PINECONE_API_KEY'),     
    environment=os.getenv('PINECONE_ENVIRONMENT')
)

AttributeError: init is no longer a top-level attribute of the pinecone package.

Please create an instance of the Pinecone class instead.

Example:

    import os
    from pinecone import Pinecone, ServerlessSpec

    pc = Pinecone(
        api_key=os.environ.get("PINECONE_API_KEY")
    )

    # Now do stuff
    if 'my_index' not in pc.list_indexes().names():
        pc.create_index(
            name='my_index', 
            dimension=1536, 
            metric='euclidean',
            spec=ServerlessSpec(
                cloud='aws',
                region='us-west-2'
            )
        )



In [ ]:
import openai
from langchain.embeddings.openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(openai_api_key=openai.api_key)

In [ ]:
def get_similar_docs(query,namespace,num_sources=5,score=False):
  index = Pinecone.from_existing_index(index_name, embeddings, namespace = namespace)
  if score:
    similar_docs = index.similarity_search_with_score(query, k=num_sources, namespace=namespace)
  else:
    similar_docs = index.similarity_search(query,k=num_sources, namespace=namespace)
  return similar_docs

In [ ]:
def get_answer(query, namespace):
  from langchain.llms import OpenAI
  llm = OpenAI(model_name=model)
  chain = load_qa_chain(llm, chain_type="stuff")
  similar_docs_list = get_similar_docs(query, namespace=namespace)
  return chain.run(input_documents=similar_docs_list, question=query)

In [ ]:
answer=get_answer("Who is Montes?", namespace)
print(answer)